# 08 - Demostración final de predicción de AKI

## Objetivo
El objetivo de este notebook fue mostrar de forma práctica cómo utilizar los modelos entrenados para estimar la probabilidad de desarrollar lesión renal aguda a partir de nuevos datos clínicos simulados. Esta fase tuvo una finalidad demostrativa y didáctica: ilustrar el funcionamiento del pipeline final, enseñar cómo se introducen nuevos casos y evidenciar que la salida del sistema depende tanto del modelo como del umbral de decisión utilizado. No representa una herramienta lista para uso clínico real, sino una prueba de concepto basada en los modelos desarrollados en el TFG.

A diferencia de los notebooks anteriores, en esta etapa no se desarrollaron nuevas tareas de limpieza, entrenamiento o evaluación formal. Su propósito fue servir como cierre aplicado del proyecto, mostrando cómo se traducen los resultados del TFG en una predicción individual sobre casos ficticios.


### 1. Importar librerías
Se importaron las librerías necesarias para:
- manipulación de datos (`pandas`, `numpy`)
- carga de modelos entrenados (`joblib`)

Estas herramientas eran suficientes para reconstruir el escenario de inferencia, es decir, el proceso mediante el cual un modelo ya entrenado recibe nuevos datos y devuelve una predicción.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

FIGURES_DIR="../figures"
RESULTS_DIR="../results"

os.makedirs(FIGURES_DIR,exist_ok=True)
os.makedirs(RESULTS_DIR,exist_ok=True)

print("Carpetas de figures y results listas.")

Carpetas de figures y results listas.


### 2. Carga del modelo entrenado
Se cargó como modelo principal de demostración la regresión logística entrenada en el notebook 05. De forma adicional, también se cargó el Random Forest para comparar el comportamiento de ambos enfoques sobre los mismos casos ficticios.

La regresión logística fue seleccionada como modelo principal de demo porque:
- es más interpretable,
- mostró mejor recall bajo el umbral estándar,
- y genera una narrativa más clara para explicar casos individuales.

El Random Forest se incorporó de forma complementaria para mostrar que dos modelos distintos pueden producir salidas diferentes ante el mismo perfil clínico.

Este paso permitió reutilizar directamente los modelos entrenados y cerrar el pipeline del proyecto mostrando la fase de inferencia individual.


In [2]:
# Primero cargamos la regresión logística como modelo principal de demo
logreg_model = joblib.load("../models/logreg_pipeline.pkl")

print("Modelo de regresión logística cargado correctamente.")

Modelo de regresión logística cargado correctamente.


In [3]:
# Luego cargamos el Random Forest
rf_model = joblib.load("../models/rf_pipeline.pkl")

print("Modelo Random Forest cargado correctamente.")

Modelo Random Forest cargado correctamente.


### 3. Definición del orden correcto de variables de entrada
Se definió explícitamente el conjunto de variables que debían recibir los modelos:

- `Sexo`
- `EdadAproxIngreso`
- `Edad_ge_65`
- `Potasio_mmol_L`
- `Sodio_mmol_L`
- `BaselineCreatinina`
- `AINE`
- `IECA`
- `ARAII`
- `DIURETICO`
- `VANCOMICINA`
- `AMINOGLUCOSIDO`
- `N_Farmacos_Riesgo`

**Justificación**

El modelo solo puede generar predicciones válidas si los nuevos datos se introducen con exactamente las mismas variables y en el mismo orden que el conjunto utilizado en el entrenamiento.

**Interpretación**

Este paso es metodológicamente esencial porque garantiza la consistencia entre el dataset de modelado y los nuevos casos introducidos en la demostración.


In [4]:
feature_cols = [
    "Sexo",
    "EdadAproxIngreso",
    "Edad_ge_65",
    "Potasio_mmol_L",
    "Sodio_mmol_L",
    "BaselineCreatinina",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO",
    "N_Farmacos_Riesgo"
]

feature_cols

['Sexo',
 'EdadAproxIngreso',
 'Edad_ge_65',
 'Potasio_mmol_L',
 'Sodio_mmol_L',
 'BaselineCreatinina',
 'AINE',
 'IECA',
 'ARAII',
 'DIURETICO',
 'VANCOMICINA',
 'AMINOGLUCOSIDO',
 'N_Farmacos_Riesgo']

### 4. Crear casos clínicos ficticios de ejemplo
Se construyeron tres perfiles simulados para la demostración:

- **Caso 1**: perfil de bajo riesgo
- **Caso 2**: perfil de riesgo elevado intermedio
- **Caso 3**: perfil de riesgo elevado más marcado

El uso de casos ficticios permite ilustrar el comportamiento del modelo sin utilizar pacientes reales ni depender de observaciones concretas del dataset. Además, facilita construir ejemplos pedagógicos con diferentes niveles esperados de riesgo.

La selección de perfiles contrastados permitió evaluar si el modelo reaccionaba de forma coherente ante cambios en edad, función renal basal y carga farmacológica.


In [5]:
demo_cases = pd.DataFrame([
    {
        "Sexo": 0,
        "EdadAproxIngreso": 35,
        "Edad_ge_65": 0,
        "Potasio_mmol_L": 4.2,
        "Sodio_mmol_L": 140.0,
        "BaselineCreatinina": 0.7,
        "AINE": 0,
        "IECA": 0,
        "ARAII": 0,
        "DIURETICO": 0,
        "VANCOMICINA": 0,
        "AMINOGLUCOSIDO": 0,
        "N_Farmacos_Riesgo": 0
    },
    {
        "Sexo": 1,
        "EdadAproxIngreso": 68,
        "Edad_ge_65": 1,
        "Potasio_mmol_L": 4.8,
        "Sodio_mmol_L": 137.0,
        "BaselineCreatinina": 1.3,
        "AINE": 1,
        "IECA": 1,
        "ARAII": 0,
        "DIURETICO": 1,
        "VANCOMICINA": 0,
        "AMINOGLUCOSIDO": 0,
        "N_Farmacos_Riesgo": 3
    },
    {
        "Sexo": 0,
        "EdadAproxIngreso": 79,
        "Edad_ge_65": 1,
        "Potasio_mmol_L": 5.1,
        "Sodio_mmol_L": 134.0,
        "BaselineCreatinina": 1.9,
        "AINE": 1,
        "IECA": 1,
        "ARAII": 0,
        "DIURETICO": 1,
        "VANCOMICINA": 1,
        "AMINOGLUCOSIDO": 0,
        "N_Farmacos_Riesgo": 4
    }
])

demo_cases

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo
0,0,35,0,4.2,140.0,0.7,0,0,0,0,0,0,0
1,1,68,1,4.8,137.0,1.3,1,1,0,1,0,0,3
2,0,79,1,5.1,134.0,1.9,1,1,0,1,1,0,4


### 5. Verificación de estructura de entrada
Antes de generar predicciones, se reordenaron las columnas de los casos ficticios para asegurar que coincidieran exactamente con `feature_cols`.
Incluso si las variables presentes son correctas, un orden incorrecto puede invalidar la predicción en modelos entrenados con pipelines estructurados.

Este paso actúa como una comprobación de integridad final antes de pasar los datos al modelo


In [6]:
demo_cases = demo_cases[feature_cols].copy()

print("Dimensiones del dataset de demo:", demo_cases.shape)
demo_cases

Dimensiones del dataset de demo: (3, 13)


,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo
0,0,35,0,4.2,140.0,0.7,0,0,0,0,0,0,0
1,1,68,1,4.8,137.0,1.3,1,1,0,1,0,0,3
2,0,79,1,5.1,134.0,1.9,1,1,0,1,1,0,4


### 6. Predicción con la regresión logística
Se utilizaron los casos ficticios como entrada de la regresión logística para obtener:

- las probabilidades estimadas de AKI
- y las predicciones binarias correspondientes.

In [7]:
logreg_pred = logreg_model.predict(demo_cases)
logreg_proba = logreg_model.predict_proba(demo_cases)[:, 1]

demo_results_logreg = demo_cases.copy()
demo_results_logreg["Probabilidad_AKI_LogReg"] = logreg_proba
demo_results_logreg["Prediccion_AKI_LogReg"] = logreg_pred

demo_results_logreg

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo,Probabilidad_AKI_LogReg,Prediccion_AKI_LogReg
0,0,35,0,4.2,140.0,0.7,0,0,0,0,0,0,0,0.012742,0
1,1,68,1,4.8,137.0,1.3,1,1,0,1,0,0,3,0.761136,1
2,0,79,1,5.1,134.0,1.9,1,1,0,1,1,0,4,0.928931,1


#### Resultados principales
- **Caso 1**: probabilidad = 0.013, predicción = 0
- **Caso 2**: probabilidad = 0.761, predicción = 1
- **Caso 3**: probabilidad = 0.929, predicción = 1

### Justificación
Obtener simultáneamente probabilidades y predicciones binarias permite mostrar la diferencia entre:
- la estimación continua del riesgo,
- y la decisión final resultante del umbral de clasificación.

#### Interpretación
La regresión logística respondió de forma coherente al gradiente de riesgo representado por los casos simulados. El caso de bajo riesgo recibió una probabilidad muy baja, mientras que los perfiles con mayor edad, mayor creatinina basal y mayor exposición farmacológica recibieron probabilidades claramente elevadas.

Este comportamiento es consistente con lo observado en notebooks anteriores y refuerza la utilidad de la regresión logística como modelo interpretable para demostraciones individuales.

### 7. Interpretación básica de las predicciones
Se presentaron las predicciones de la regresión logística en formato narrativo, caso por caso, mostrando:

- edad
- creatinina basal
- número de fármacos de riesgo
- probabilidad estimada
- categoría de riesgo
- y predicción binaria

#### Justificación
La presentación narrativa hace que los resultados sean más fáciles de comprender y comunicar, especialmente en el contexto de una defensa oral o de una memoria académica.

#### Interpretación
Esta forma de salida mostró que el modelo no solo genera una etiqueta binaria, sino también una estimación probabilística que puede ser clínicamente más informativa que una respuesta dicotómica aislada.


In [8]:
for i, row in demo_results_logreg.iterrows():
    print(f"Caso {i+1}")
    print(f"Probabilidad estimada de AKI (Regresión Logística): {row['Probabilidad_AKI_LogReg']:.3f}")
    print(f"Predicción binaria: {int(row['Prediccion_AKI_LogReg'])}")
    print("-" * 50)

Caso 1
Probabilidad estimada de AKI (Regresión Logística): 0.013
Predicción binaria: 0
--------------------------------------------------
Caso 2
Probabilidad estimada de AKI (Regresión Logística): 0.761
Predicción binaria: 1
--------------------------------------------------
Caso 3
Probabilidad estimada de AKI (Regresión Logística): 0.929
Predicción binaria: 1
--------------------------------------------------


### 8. Añadir una etiqueta interpretativa sencilla
Se definió una categorización pedagógica del riesgo basada en la probabilidad estimada por la regresión logística:

- menor de 0.10 → **Bajo**
- entre 0.10 y 0.30 → **Intermedio**
- igual o superior a 0.30 → **Alto**

**Justificación**

Esta clasificación no tuvo la intención de crear un sistema clínico validado, sino de facilitar una lectura intuitiva de la probabilidad generada por el modelo en el contexto de una demostración.

**Interpretación**

La categorización permitió distinguir de forma comprensible entre un perfil claramente bajo riesgo y perfiles con probabilidad elevada de AKI, reforzando el valor didáctico del notebook.


In [9]:
def categorize_risk(prob):
    if prob < 0.10:
        return "Bajo"
    elif prob < 0.30:
        return "Intermedio"
    else:
        return "Alto"

demo_results_logreg["Categoria_Riesgo_LogReg"] = demo_results_logreg["Probabilidad_AKI_LogReg"].apply(categorize_risk)

demo_results_logreg[[
    "Probabilidad_AKI_LogReg",
    "Prediccion_AKI_LogReg",
    "Categoria_Riesgo_LogReg"
]]

,Probabilidad_AKI_LogReg,Prediccion_AKI_LogReg,Categoria_Riesgo_LogReg
0,0.012742,0,Bajo
1,0.761136,1,Alto
2,0.928931,1,Alto


### 9. Demo opcional con Random Forest
Se repitió la predicción sobre los mismos casos utilizando el Random Forest, obteniendo sus probabilidades estimadas y sus predicciones binarias con el umbral estándar de 0.5.

In [10]:
rf_pred_default = rf_model.predict(demo_cases)
rf_proba = rf_model.predict_proba(demo_cases)[:, 1]

demo_results_rf = demo_cases.copy()
demo_results_rf["Probabilidad_AKI_RF"] = rf_proba
demo_results_rf["Prediccion_AKI_RF_0_5"] = rf_pred_default

demo_results_rf

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo,Probabilidad_AKI_RF,Prediccion_AKI_RF_0_5
0,0,35,0,4.2,140.0,0.7,0,0,0,0,0,0,0,0.00,0
1,1,68,1,4.8,137.0,1.3,1,1,0,1,0,0,3,0.09,0
2,0,79,1,5.1,134.0,1.9,1,1,0,1,1,0,4,0.10,0


#### Resultados principales
- **Caso 1**: probabilidad = 0.00
- **Caso 2**: probabilidad = 0.09
- **Caso 3**: probabilidad = 0.10

En todos los casos, la predicción con threshold 0.5 fue negativa.

#### Justificación
El objetivo de esta sección era comparar cómo dos modelos distintos podían producir respuestas muy diferentes sobre los mismos perfiles.

#### Interpretación
El Random Forest mostró nuevamente un comportamiento muy conservador, incluso en perfiles clínicamente más complejos. Aunque incrementó ligeramente la probabilidad en los casos de mayor riesgo, no alcanzó valores suficientes como para emitir predicciones positivas bajo el umbral estándar.

Este resultado fue coherente con la evaluación realizada en el notebook 06.

### 10. Aplicar un umbral alternativo al Random Forest
Dado que en el notebook 06 se había observado que el Random Forest mejoraba su utilidad práctica con thresholds menores, se aplicó un umbral alternativo de 0.20 sobre los casos de demostración.

In [11]:
rf_threshold = 0.20
rf_pred_02 = (rf_proba >= rf_threshold).astype(int)

demo_results_rf["Prediccion_AKI_RF_0_2"] = rf_pred_02

demo_results_rf[[
    "Probabilidad_AKI_RF",
    "Prediccion_AKI_RF_0_5",
    "Prediccion_AKI_RF_0_2"
]]

,Probabilidad_AKI_RF,Prediccion_AKI_RF_0_5,Prediccion_AKI_RF_0_2
0,0.00,0,0
1,0.09,0,0
2,0.10,0,0


#### Resultado
Ninguno de los tres casos superó este umbral, por lo que las predicciones continuaron siendo negativas.

#### Justificación
Esta prueba permitió comprobar si los perfiles simulados alcanzaban un nivel de riesgo suficiente para activar al Random Forest incluso bajo un umbral más bajo.

#### Interpretación
El resultado sugiere que, en estos ejemplos concretos, el Random Forest siguió siendo más conservador que la regresión logística, incluso utilizando un umbral alternativo. Esto refuerza la idea de que ambos modelos no solo tienen distinto rendimiento global, sino también distinta sensibilidad en predicción individual.


### 11. Comparación final entre modelos en los casos de demo
Se construyó una tabla final (`final_demo`) reuniendo, para cada caso:

- variables de entrada
- probabilidad y predicción de la regresión logística
- categoría de riesgo según la regresión logística
- probabilidad del Random Forest
- predicción del Random Forest con threshold 0.5
- y predicción del Random Forest con threshold 0.2

#### Justificación
La comparación conjunta permite visualizar de forma compacta y directa las diferencias de comportamiento entre modelos.

#### Interpretación
La tabla mostró con claridad que:
- la regresión logística distinguía mejor entre bajo y alto riesgo en estos casos de demo,
- mientras que el Random Forest generaba probabilidades más contenidas y mantenía una salida más conservadora


In [12]:
final_demo = demo_cases.copy()

final_demo["Prob_LogReg"] = logreg_proba
final_demo["Pred_LogReg"] = logreg_pred
final_demo["Cat_Riesgo_LogReg"] = final_demo["Prob_LogReg"].apply(categorize_risk)

final_demo["Prob_RF"] = rf_proba
final_demo["Pred_RF_0_5"] = rf_pred_default
final_demo["Pred_RF_0_2"] = rf_pred_02

final_demo

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo,Prob_LogReg,Pred_LogReg,Cat_Riesgo_LogReg,Prob_RF,Pred_RF_0_5,Pred_RF_0_2
0,0,35,0,4.2,140.0,0.7,0,0,0,0,0,0,0,0.012742,0,Bajo,0.00,0,0
1,1,68,1,4.8,137.0,1.3,1,1,0,1,0,0,3,0.761136,1,Alto,0.09,0,0
2,0,79,1,5.1,134.0,1.9,1,1,0,1,1,0,4,0.928931,1,Alto,0.10,0,0


In [13]:
final_demo.to_csv(f"{RESULTS_DIR}/results_08_demo_predictions.csv", index=False)

### 12. Presentación más narrativa de cada caso
Se generó una salida narrativa individual para cada caso, integrando variables clave con las probabilidades y predicciones obtenidas.

In [14]:
for i, row in final_demo.iterrows():
    print(f"\n=== Caso {i+1} ===")
    print(f"Edad: {row['EdadAproxIngreso']} años")
    print(f"Creatinina basal: {row['BaselineCreatinina']}")
    print(f"Número de fármacos de riesgo: {row['N_Farmacos_Riesgo']}")
    print(f"Probabilidad AKI - Regresión Logística: {row['Prob_LogReg']:.3f}")
    print(f"Categoría de riesgo - Regresión Logística: {row['Cat_Riesgo_LogReg']}")
    print(f"Probabilidad AKI - Random Forest: {row['Prob_RF']:.3f}")
    print(f"Predicción RF con threshold 0.5: {int(row['Pred_RF_0_5'])}")
    print(f"Predicción RF con threshold 0.2: {int(row['Pred_RF_0_2'])}")


=== Caso 1 ===
Edad: 35 años
Creatinina basal: 0.7
Número de fármacos de riesgo: 0
Probabilidad AKI - Regresión Logística: 0.013
Categoría de riesgo - Regresión Logística: Bajo
Probabilidad AKI - Random Forest: 0.000
Predicción RF con threshold 0.5: 0
Predicción RF con threshold 0.2: 0

=== Caso 2 ===
Edad: 68 años
Creatinina basal: 1.3
Número de fármacos de riesgo: 3
Probabilidad AKI - Regresión Logística: 0.761
Categoría de riesgo - Regresión Logística: Alto
Probabilidad AKI - Random Forest: 0.090
Predicción RF con threshold 0.5: 0
Predicción RF con threshold 0.2: 0

=== Caso 3 ===
Edad: 79 años
Creatinina basal: 1.9
Número de fármacos de riesgo: 4
Probabilidad AKI - Regresión Logística: 0.929
Categoría de riesgo - Regresión Logística: Alto
Probabilidad AKI - Random Forest: 0.100
Predicción RF con threshold 0.5: 0
Predicción RF con threshold 0.2: 0


#### Resultados
- **Caso 1** fue interpretado de forma coherente por ambos modelos como un perfil de bajo riesgo.
- **Caso 2** y **Caso 3** fueron clasificados como de alto riesgo por la regresión logística, mientras que el Random Forest mantuvo salida negativa en ambos casos.
- El aumento progresivo de la probabilidad en la regresión logística fue consistente con el empeoramiento del perfil clínico.

#### Justificación
Esta presentación favorece una lectura más cercana a un escenario clínico o de explicación oral, donde no basta con mostrar una tabla, sino que conviene narrar el caso y su resultado.

#### Interpretación
El comportamiento observado refuerza la utilidad de la regresión logística como herramienta demostrativa y como modelo más fácil de comunicar en contextos individuales.



### 13. Ejemplo de entrada manual de un nuevo caso
Se creó un cuarto caso ficticio de forma individual, simulando cómo introducir manualmente los datos de un nuevo ingreso hospitalario.

#### Justificación
Este ejemplo permite mostrar que el sistema puede utilizarse no solo con lotes de casos ya definidos, sino también con una única entrada manual, lo que refuerza la demostración técnica del pipeline.

#### Interpretación
Este paso acerca la demo a un escenario de uso práctico, mostrando cómo se podría introducir un nuevo perfil y obtener su estimación de riesgo.


In [15]:
nuevo_caso = pd.DataFrame([{
    "Sexo": 1,
    "EdadAproxIngreso": 72,
    "Edad_ge_65": 1,
    "Potasio_mmol_L": 4.9,
    "Sodio_mmol_L": 136.0,
    "BaselineCreatinina": 1.4,
    "AINE": 0,
    "IECA": 1,
    "ARAII": 0,
    "DIURETICO": 1,
    "VANCOMICINA": 0,
    "AMINOGLUCOSIDO": 0,
    "N_Farmacos_Riesgo": 2
}])

nuevo_caso = nuevo_caso[feature_cols]

nuevo_caso

,Sexo,EdadAproxIngreso,Edad_ge_65,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,N_Farmacos_Riesgo
0,1,72,1,4.9,136.0,1.4,0,1,0,1,0,0,2


In [16]:
nuevo_caso.to_csv(f"{RESULTS_DIR}/results_08_new_case_input.csv", index=False)

### 14. Predicción del nuevo caso
Se generaron predicciones para este nuevo caso utilizando ambos modelos.

In [17]:
prob_nuevo_logreg = logreg_model.predict_proba(nuevo_caso)[:, 1][0]
pred_nuevo_logreg = logreg_model.predict(nuevo_caso)[0]

prob_nuevo_rf = rf_model.predict_proba(nuevo_caso)[:, 1][0]
pred_nuevo_rf_05 = rf_model.predict(nuevo_caso)[0]
pred_nuevo_rf_02 = int(prob_nuevo_rf >= 0.20)

print("=== Nuevo caso ===")
print(f"Probabilidad AKI - Regresión Logística: {prob_nuevo_logreg:.3f}")
print(f"Predicción LogReg: {pred_nuevo_logreg}")
print(f"Categoría de riesgo LogReg: {categorize_risk(prob_nuevo_logreg)}")
print(f"Probabilidad AKI - Random Forest: {prob_nuevo_rf:.3f}")
print(f"Predicción RF con threshold 0.5: {pred_nuevo_rf_05}")
print(f"Predicción RF con threshold 0.2: {pred_nuevo_rf_02}")

=== Nuevo caso ===
Probabilidad AKI - Regresión Logística: 0.447
Predicción LogReg: 0
Categoría de riesgo LogReg: Alto
Probabilidad AKI - Random Forest: 0.070
Predicción RF con threshold 0.5: 0
Predicción RF con threshold 0.2: 0


In [18]:
nuevo_caso_resultado = pd.DataFrame([{
    "Probabilidad_AKI_LogReg": prob_nuevo_logreg,
    "Prediccion_LogReg": pred_nuevo_logreg,
    "Categoria_Riesgo_LogReg": categorize_risk(prob_nuevo_logreg),
    "Probabilidad_AKI_RF": prob_nuevo_rf,
    "Prediccion_RF_0_5": pred_nuevo_rf_05,
    "Prediccion_RF_0_2": pred_nuevo_rf_02
}])

nuevo_caso_resultado.to_csv(f"{RESULTS_DIR}/results_08_new_case_prediction.csv", index=False)

#### Resultados
- **Regresión Logística**
  - probabilidad = 0.447
  - predicción binaria = 0
  - categoría de riesgo = Alto

- **Random Forest**
  - probabilidad = 0.070
  - predicción con threshold 0.5 = 0
  - predicción con threshold 0.2 = 0

#### Justificación
Este resultado fue particularmente útil para mostrar la diferencia entre:
- probabilidad estimada,
- clasificación binaria final,
- y categoría interpretativa.

#### Interpretación
El nuevo caso fue especialmente interesante porque la regresión logística le asignó una probabilidad relativamente alta (0.447), aunque sin superar el umbral estándar de 0.5. Esto mostró de forma muy clara que la probabilidad continua del modelo puede contener información relevante incluso cuando la predicción binaria final es negativa.

Desde una perspectiva explicativa, este ejemplo fue muy valioso porque permitió ilustrar que:
- no todos los casos con riesgo elevado percibido por el modelo se convierten automáticamente en positivos binarios,
- y que la interpretación de probabilidades puede aportar una lectura más rica que la etiqueta final por sí sola.


### 15. Conclusiones del notebook

Este notebook mostró cómo utilizar los modelos entrenados para generar predicciones sobre nuevos casos clínicos simulados.

La demostración permitió comprobar que:

- el modelo puede recibir como entrada un conjunto reducido de variables clínicas estructuradas,
- puede generar una probabilidad estimada de desarrollo de AKI,
- y puede convertir esa probabilidad en una predicción binaria según el umbral elegido.

Asimismo, se observó que la salida final depende no solo del modelo, sino también del threshold de clasificación adoptado, especialmente en el caso del Random Forest.

Esta demostración debe entenderse como una prueba de concepto técnica y académica. No representa una herramienta validada para uso clínico real ni sustituye el juicio médico.

Este notebook cerró el pipeline del TFG mostrando cómo se aplican los modelos entrenados a nuevos casos clínicos simulados.

### Hallazgos principales
- la regresión logística generó probabilidades coherentes con el gradiente de riesgo clínico de los casos simulados,
- distinguió de forma clara entre perfiles de bajo y alto riesgo,
- y resultó especialmente útil como modelo demostrativo por su interpretabilidad y por su sensibilidad en predicción individual.

Por su parte:
- el Random Forest mantuvo una salida mucho más conservadora,
- incluso en perfiles con creatinina basal elevada, edad avanzada y carga farmacológica alta,
- lo que fue consistente con su comportamiento observado en la fase de evaluación.

### Conclusión metodológica principal
La demostración final mostró que el sistema desarrollado es capaz de recibir un conjunto estructurado de variables clínicas y producir una estimación individual de riesgo de AKI. Aunque esta prueba de concepto no representa una herramienta clínica validada, sí demuestra la viabilidad técnica y metodológica del pipeline completo desarrollado en el TFG, desde la extracción de datos hasta la predicción sobre casos nuevos.
